# Kaggle evaluation run — finalproject_cpe027

Runs the Tier 1 (ACL6060 + seeded DEMAND noise) and Tier 2 (unmodified M3AV) ASR evaluation
on Kaggle (GPU) using preprocessed local data. No preprocessing is redone here.

- Tiers are executed and reported separately; they are never pooled.
- Checkpoints are retained per tier; this notebook starts fresh (`--no-resume`).
- Whisper: `base`, `cuda`, `float16`, `en` (Tier 1). DSP/decoding settings come from the runner.
- Assumed mounts: `/kaggle/input/acl6060`, `/kaggle/input/demand`, `/kaggle/input/m3av`.

In [1]:
!nvidia-smi

Thu Nov  5 20:11:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!df -h

Filesystem        Size  Used Avail Use% Mounted on
overlay           8.0T  7.1T  967G  89% /
tmpfs              64M     0   64M   0% /dev
shm                14G     0   14G   0% /dev/shm
/dev/sda1         122G  5.0G  117G   5% /opt/bin
/dev/loop2         20G   80K   20G   1% /kaggle/lib
kagglesdsdata     1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
kagglesdsdata     1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand
/dev/mapper/snap  8.0T  7.1T  967G  89% /etc/hosts
tmpfs             4.0K     0  4.0K   0% /proc/acpi


In [3]:
!ls /kaggle/input/*

from pathlib import Path

for p in sorted(Path("/kaggle/input").iterdir()):
    print(p.name, "->", sorted(c.name for c in p.iterdir()) if p.is_dir() else "(file)")

leonpascual1
datasets -> ['leonpascual1']


In [9]:
!pip install -q faster-whisper av scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic

In [5]:
import os

os.environ["WHISPER_MODEL"] = "base"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["WHISPER_THREADS"] = "4"
os.environ["WHISPER_LANGUAGE"] = "en"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/models"
os.environ["EVAL_DATA_ROOT"] = "/kaggle/input/acl6060"

In [10]:
from importlib.metadata import version

for pkg in ["ctranslate2", "faster-whisper", "av", "scipy", "numpy", "rapidfuzz", "matplotlib", "pydantic", "pydantic_settings"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception:
        print(f"{pkg}: unknown")

try:
    import torch
    print("torch:", torch.__version__)
except Exception:
    print("torch: not installed")

ctranslate2: 4.8.5
faster-whisper: 1.2.1
av: 19.0.2
scipy: 1.16.3
numpy: 2.1.3
rapidfuzz: 3.14.6
matplotlib: 3.10.0
pydantic: 2.14.1
pydantic_settings: 2.16.2
torch: 2.11.0+cu130


In [7]:
import os
import sys

sys.path.insert(0, "/kaggle/working/finalproject_cpe027/backend")

from pathlib import Path

from app import eval_data

CLIP_ROOTS = sorted(
    str(p / "M3AV" / "extracted")
    for p in Path("/kaggle/input").iterdir()
    if p.is_dir() and (p / "M3AV" / "extracted").is_dir()
)
os.environ["M3AV_CLIP_ROOT"] = CLIP_ROOTS[0]
print("ACL6060 root:", eval_data.acl6060_root("/kaggle/input/acl6060"))
print("M3AV roots:", CLIP_ROOTS)
print("DEMAND env dir:", eval_data.demand_noise_file("/kaggle/input/demand/DKITCHEN_16k"))


ModuleNotFoundError: No module named 'app'

## Smoke test

One talk / one clip. Run this first and check WER + RTF before any full run.

In [ ]:
%cd /kaggle/working/finalproject_cpe027/backend

!python -m app.eval_runner --tier all --limit 1 --conditions arm_a,hp --snrs 0 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root /kaggle/input/acl6060 --demand-root /kaggle/input/demand --m3av-root $M3AV_CLIP_ROOT --out /kaggle/working/eval_smoke


In [ ]:
import json
from pathlib import Path

t1 = json.loads(Path("/kaggle/working/eval_smoke/tier1_acl_results.json").read_text())
t2 = json.loads(Path("/kaggle/working/eval_smoke/tier2_m3av_results.json").read_text())
print("SMOKE tier1 records:", len(t1))
r = t1[0]
print(f"SMOKE WER: {r['wer']:.2%} | RTF: {r['rtf']:.3f} | seconds: {r['seconds']:.1f}")
print("SMOKE tier2 records:", len(t2))
r2 = t2[0]
print(f"SMOKE tier2 WER: {r2['wer']:.2%} | RTF: {r2['rtf']:.3f} | NSHR: {r2['nshr']}")

## Full run — Tier 1 (ACL6060 + seeded DEMAND noise)

All six conditions, SNRs `none,0,5,10,20`, DEMAND envs `DKITCHEN,DLIVING,DWASHING`. Fresh run (`--no-resume`).

In [ ]:
%cd /kaggle/working/finalproject_cpe027/backend

!python -m app.eval_runner --tier 1 --split eval --conditions arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa --snrs none,0,5,10,20 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root /kaggle/input/acl6060 --demand-root /kaggle/input/demand --out /kaggle/working/batches/tier1_eval --no-resume

## Full run — Tier 2 (M3AV), batched per duration bucket

In [ ]:
from types import SimpleNamespace

from app import eval_data, eval_runner

CONDITIONS = "arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa"
BUCKETS = ["<5m", "5-10m", "10-20m", ">=20m"]

for clip_root in CLIP_ROOTS:
    for bucket in BUCKETS:
        selected = [c for c in eval_data.load_m3av_clips(clip_root) if eval_data.duration_bucket(c.duration_seconds) == bucket]
        if not selected:
            continue
        out_dir = "/kaggle/working/batches/tier2_" + bucket.replace("<", "lt_").replace(">", "gt_").replace("+", "p") + "_" + Path(clip_root).parent.parent.name
        orig_loader = eval_runner.load_m3av_clips
        eval_runner.load_m3av_clips = lambda data_root=None, _sel=selected: _sel
        args = SimpleNamespace(
            tier="2",
            conditions=CONDITIONS,
            m3av_root=clip_root,
            data_root=None,
            out=out_dir,
            limit=None,
            resume=False,
        )
        try:
            records, summary = eval_runner.run_tier2(args)
        finally:
            eval_runner.load_m3av_clips = orig_loader
        print(f"tier2 {Path(clip_root).name} bucket {bucket}: {len(records)} records -> {out_dir}")


## Merge batch outputs into final results

In [ ]:
%cd /kaggle/working/finalproject_cpe027

!python merge_eval_results.py --batch-dir /kaggle/working/batches --out /kaggle/working/final

In [ ]:
from pathlib import Path

names = [
    "tier1_acl_results.json",
    "tier1_acl_results.csv",
    "tier1_acl_summary.json",
    "tier2_m3av_results.json",
    "tier2_m3av_results.csv",
    "tier2_m3av_summary.json",
]
for name in names:
    p = Path("/kaggle/working/final") / name
    print(name, "OK" if p.is_file() else f"MISSING: {p}")